# Initialize

In [1]:
import yaml
import pandas as pd
from pathlib import Path
import os
from transformers import AlbertForSequenceClassification, AlbertTokenizerFast
from transformers import PreTrainedTokenizerBase
import numpy as np

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").is_dir() and (candidate / "configs" / "config.yaml").exists():
            return candidate

    raise FileNotFoundError("未找到包含 src/ 和 config.yaml 的项目根目录")


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path.cwd()}")

Project root: C:\Personal programs\sentence_error_detection_project
Working directory: C:\Personal programs\sentence_error_detection_project


## load data

In [3]:
with open(r"configs/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

In [4]:
sen_df = pd.read_csv(Path(config["data"]["processed_dir"]) / "toxicity/sentences.csv")

# Analysis

## basic imformation

In [5]:
print(f"Number of sentences: {len(sen_df)}")

Number of sentences: 2848


In [6]:
print(f"positive toxicity sentences: {len(sen_df[sen_df['label'] == 1])}")
print(f"positive toxicity sentences rate : {len(sen_df[sen_df['label'] == 1]) / len(sen_df)}")

positive toxicity sentences: 603
positive toxicity sentences rate : 0.21172752808988765


# analyse token lengths distribution

In [25]:
def analyse_token_lengths(
    texts: list[str],
    tokenizer: PreTrainedTokenizerBase,
) -> None:
    lengths = [
        len(
            tokenizer(
                text,
                add_special_tokens=True,
                truncation=False,
            )["input_ids"]
        )
        for text in texts
    ]

    percentiles = [50, 75, 90, 95, 99, 100]

    for percentile in percentiles:
        value = np.percentile(lengths, percentile)
        print(f"{percentile:>3}% percentile: {value:.0f} tokens")

    print(f"Mean: {np.mean(lengths):.2f}")
    print(f"Maximum: {max(lengths)}")

In [27]:
analyse_token_lengths(sen_df["sentence"].tolist(), AlbertTokenizerFast.from_pretrained("albert-base-v2"))

 50% percentile: 24 tokens
 75% percentile: 31 tokens
 90% percentile: 39 tokens
 95% percentile: 45 tokens
 99% percentile: 59 tokens
100% percentile: 107 tokens
Mean: 25.75
Maximum: 107
